In [ ]:
# !pip install openai==1.55.2 python-dotenv==1.0.1 html-to-markdown==2.5.0

In [ ]:
import os
import sys
import time
import json
from pydantic import BaseModel

import nest_asyncio
nest_asyncio.apply()

from openai import OpenAI
client = OpenAI()

from dotenv import load_dotenv
load_dotenv()

In [ ]:
sys.path.append("..")
from scripts.agent_playground import AgentPlayground

playground = AgentPlayground()
# playground.clean()

In [ ]:
# User Input
TASK = "Elk Detection"
DATASET_PATH = "../.data/elk"
MODEL_ARCHITECTURE = "YOLOv8"
MODEL_WEIGHTS = "../data/yolo-finetune-elk/10_epochs_224_imgsz/weights/best.pt"
MODEL_CODE = "../scripts/yolo_minimal.py"
DEVICE = "MAX78000"
CUSTOM_INSTRUCTIONS = """
    You do not need to preserve the part of the C2f that splits the input into chunks.
""".strip()

In [ ]:
# Transfer input to playground
PLAYGROUND_WEIGHTS = MODEL_WEIGHTS.split('/')[-1]
playground.copy_to_playground(MODEL_WEIGHTS, PLAYGROUND_WEIGHTS)
PLAYGROUND_CODE = MODEL_CODE.split('/')[-1]
playground.copy_to_playground(MODEL_CODE, PLAYGROUND_CODE)
PLAYGROUND_DATA = "data"
playground.copy_to_playground(DATASET_PATH, PLAYGROUND_DATA)

In [ ]:
# Agent roles
FIRMWARE_ENGINEER = f"""
You are a precise and careful Firmware Engineer working on a team to take a given model, ML task, dataset, plus target device and adapt the model to run on that target device.
The task is "{TASK}", the model architecture is "{MODEL_ARCHITECTURE}" (with model weights in "{PLAYGROUND_WEIGHTS}" and code in "{PLAYGROUND_CODE}"), the data is stored in the "{PLAYGROUND_DATA}" folder, and the target device is "{DEVICE}". Documentation belongs in the "docs" folder.
You are in charge of the hardware-software interface which means you must understand the hardware constraints of the {DEVICE} (available weight and data memory, supported operations) and setup the software side of the project (SDK/library code, official device training/code-synthesis framework if available) using best practices including python virtual environment(s) using "python -m venv ./venv".
{CUSTOM_INSTRUCTIONS}
""".strip()
ML_ENGINEER = f"""
You are a precise and careful ML Engineer working on a team to take a given model, ML task, dataset, plus target device and adapt the model to run on that target device.
The task is "{TASK}", the model architecture is "{MODEL_ARCHITECTURE}" (with model weights in "{PLAYGROUND_WEIGHTS}" and code in "{PLAYGROUND_CODE}"), the data is stored in the "{PLAYGROUND_DATA}" folder, and the target device is "{DEVICE}". Documentation belongs in the "docs" folder.
You are in charge for taking the hardware constraints identified by the Firmware Engineer Agent and the core components identified by the ML Scientist Agent and writing the appropriate hardware adapted model architecture specification and code implementation.
{CUSTOM_INSTRUCTIONS}
""".strip()
ML_SCIENTIST = f"""
You are a precise and careful ML Scientist working on a team to take a given model, ML task, dataset, plus target device and adapt the model to run on that target device.
The task is "{TASK}", the model architecture is "{MODEL_ARCHITECTURE}" (with model weights in "{PLAYGROUND_WEIGHTS}" and code in "{PLAYGROUND_CODE}"), the data is stored in the "{PLAYGROUND_DATA}" folder, and the target device is "{DEVICE}". Documentation belongs in the "docs" folder.
You are in charge of understanding the original model architecture plus training and evaluating the modified architecture for the device.
{CUSTOM_INSTRUCTIONS}
""".strip()

In [ ]:
# Output datastructures for each section

class ResearchHardwareConstraints(BaseModel):
    datasheet_path: str
    hardware_constraints_path: str
    inference_code_synthesis_docs_path: str
    custom_model_specification_docs_path: str
    custom_dataset_docs_path: str
    examples_path: str
    summary: str

class KeyMLComponent(BaseModel):
    description: str
    code_snippet: str

class ResearchKeyMLComponents(BaseModel):
    important_architecture_features: list[KeyMLComponent]
    sources: list[str]
    notes: str

class WriteModelSpecification(BaseModel):
    compressed_model_architecture_code_path: str
    side_car_file_paths: list[str]
    sample_model_input_path: str
    randomly_initiated_model_weights_path: str

class VerifyHardwareConstraints(BaseModel):
    meets_all_constraints: bool
    errors: str | None
    feedback: str | None

class VerifyML(BaseModel):
    is_fully_valid_model: bool
    errors: str | None
    feedback: str | None

class TrainAndEvaluateModel(BaseModel):
    trained_weights_path: str
    evaluation_score: float
    evaluation_metric: float
    recommend_improvements_to_architecture: bool
    notes: str

class FinalOutput(BaseModel):
    compressed_model_path: str
    compressed_model_size_bytes: int
    inference_code_path: str
    notes: str

In [ ]:
# Tools
def format_run_command(args):
    stdout, stderr, exitcode = playground.run(args['cmd'], timeout_seconds=120)
    output = f"OUTPUT:\n{stdout}"
    if exitcode != 0:
        output += f"\nERROR (code={exitcode}):\n{stderr}"
    return output

def read_file(args):
    with playground.open(args["path"], "r") as f:
        return f.read()
    
def write_file(args):
    with playground.open(args["path"], "w") as f:
        f.write(args["text"])

TOOLS = {
    "WEB_SEARCH": {
        "spec": {
            "type": "function", 
            "function": {
                "name": "WEB_SEARCH",
                "description": "Use Google Search to find documentation, datasheets, example code, and existing solutions to problems you encounter.",
                "parameters": {
                    "type": "object",    
                    "properties": {        
                        "query": {            
                            "type": "string",            
                            "description": "The google search query",    
                        },
                    },
                    "required": ["query"],
                },
            }
        },
        "fn": lambda args: playground.search_google(args['query'], num_results=5),
    },
    "DOWNLOAD_MARKDOWN": {
        "spec": {
            "type": "function", 
            "function": {
                "name": "DOWNLOAD_MARKDOWN",
                "description": "Use the urls returned by WEB_SEARCH to download the markdown of a website/PDF. Then use the READ_FILE tool to get the entire content or the RUN_COMMAND with 'grep' to search through the output.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The url returned by WEB_SEARCH or found linked in documentation",
                        },
                        "destination_path": {
                            "type": "string",
                            "description": "The path to write the markdown to",
                        },
                    },
                    "required": ["url", "destination_path"],
                },
            }
        },
        "fn": lambda args: "success" if playground.download_url_as_markdown(args['url'], args['destination_path'], timeout_seconds=120) else "failure",
    },
    "READ_FILE": {
        "spec": {
            "type": "function", 
            "function": {
                "name": "READ_FILE",
                "description": "Gets the entire text content of a file.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "The path to read",
                        },
                    },
                    "required": ["path"],
                },
            }
        },
        "fn": lambda args: read_file(args),
    },
    "WRITE_FILE": {
        "spec": {
            "type": "function", 
            "function": {
                "name": "WRITE_FILE",
                "description": "Writes text content to a file. The parent directory must first exist.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "The path to read",
                        },
                        "text": {
                            "type": "string",
                            "description": "The content to place in the file",
                        },
                    },
                    "required": ["path", "text"],
                },
            }
        },
        "fn": lambda args: write_file(args) or "success",
    },
    "RUN_COMMAND": {
        "spec": {
            "type": "function",
            "function": {
                "name": "RUN_COMMAND",
                "description": "Run unix commands and python code, e.g., cmd=['python', 'test.py'] or cmd=['wget', 'https://example.com'] or cmd=['git', 'clone', 'https://github.com/example/example.git']. You can write a python file first using the WRITE_FILE tool.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "cmd": {
                            "type": "array",
                            "description": "first element is the command, the rest are the arguments; the usual GNU Core Utilities are available as well as 'python' (3.9)",
                            "items": {
                                "type": "string"
                            }
                        },
                    },
                    "required": ["cmd"],
                }
            }
        },
        "fn": lambda args: format_run_command(args),
    },
}

In [ ]:
from typing import TypeVar

T = TypeVar('T')

def run_with_tools(role_prompt: str, user_prompt: str, output_type: type[T], num_tool_calls = 1, tools: dict[str, dict]=TOOLS, model = "gpt-5", logfile='agent_runs.txt') -> tuple[T, int, float]:
    start_time = time.perf_counter_ns()
    total_tokens = 0
    tool_calls = []
    messages = [
        {
            "role": "system", 
            "content": f"""
{role_prompt}
At each turn, you must propose one or more tool uses until the number of remaining turns reaches zero.
When it reaches zero, produce your final answer. All paths must be specified relative to the working directory.
            """.strip()
        }
    ]
    messages.append({"role": "user", "content": user_prompt})

    try:
        for i in range(num_tool_calls):
            messages.append({"role": "system", "content": f"You have {num_tool_calls - i} turns remaining."})
            response = client.chat.completions.create(
                model=model,
                messages=messages, # type: ignore
                tools=[t["spec"] for t in tools.values()],
                tool_choice="required",
            )
            total_tokens += response.usage.total_tokens # type: ignore
            messages.append(response.choices[0].message) # type: ignore
            for call in response.choices[0].message.tool_calls: # type: ignore
                action = call.function.name # type: ignore
                tool_call_id = call.id # type: ignore
                arguments = json.loads(call.function.arguments) # type: ignore
                tool_calls.append((action, tool_call_id, arguments))
                try:
                    tool_results = tools[action]["fn"](arguments)
                except Exception as e:
                    tool_results = str(e)
                messages.append({"role": "tool", "tool_call_id": tool_call_id, "content": tool_results})
    except Exception as e:
        print(messages)
        raise e

    messages.append({"role": "system", "content": f"You have no tool uses remaining. Please answer the user prompt."})
    messages.append({"role": "user", "content": user_prompt})
    try:
        response = client.beta.chat.completions.parse(
            model=model,
            messages=messages, # type: ignore
            response_format=output_type,
        )
        total_tokens += response.usage.total_tokens # type: ignore
    except Exception as e:
        print(messages)
        raise e
    total_time = (time.perf_counter_ns() - start_time) * 0.000000001

    with open(logfile, "a") as f:
        f.write(f"-------------------------------------- New Run {time.asctime()} --------------------------------------\n")
        f.write("MESSAGES: " + json.dumps([{'role': m['role'], 'content': m['content']} for m in messages]) + "\n")
        f.write("TOOL CALLS: " + json.dumps(tool_calls) + "\n")
        f.write("TOTAL TOKENS: " + str(total_tokens) + "\n")
        f.write("TOTAL TIME: " + str(total_time) + "\n\n")

    predicted = output_type.model_validate_json(response.choices[0].message.content) # type: ignore
    return predicted, total_tokens, total_time

In [ ]:
def list_files(startpath, max_level=2):
    output = ""
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep) - 1
        if level == -1 or level > max_level:
            continue
        indent = ' ' * 4 * (level)
        output += '{}{}/\n'.format(indent, os.path.basename(root))
        subindent = ' ' * 4 * (level + 1)
        if len(files) > 10 and all(f.endswith(files[0].split('.')[-1]) for f in files):
            output += '{}{}\n'.format(subindent, f"*.{files[0].split('.')[-1]}")
        elif len(files) > 10:
            output += '{}{}\n'.format(subindent, "...")
        else:
            for f in files:
                output += '{}{}\n'.format(subindent, f)
    return output

In [ ]:
RESEARCH_HARDWARE_CONSTRAINTS_PROMPT = f"""
Please download all relevant documentation and example code to the 'docs' directory.
If there is any official model training, compression, and/or synthesis code, please set it up.
At the end, communicate your findings to the ML Engineer who will use it to draft up a model architecture.
These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

hardware_research_result, hardware_research_tokens, hardware_research_time = run_with_tools(FIRMWARE_ENGINEER, RESEARCH_HARDWARE_CONSTRAINTS_PROMPT, ResearchHardwareConstraints, num_tool_calls=10)
print(hardware_research_result, hardware_research_tokens, hardware_research_time)

In [ ]:
RESEARCH_KEY_ML_COMPONENTS_PROMPT = f"""
Please use websearch and code execution to analyze the provided model architecture ({MODEL_ARCHITECTURE}).
Use reliable research papers and inspection of the model network to identify the distinguishing features
that should be preserved if possible when compressing the model. Communicate your findings to the ML Engineer
who will use it to draft up a model architecture.
These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

ml_research_result, ml_research_tokens, ml_research_time = run_with_tools(ML_SCIENTIST, RESEARCH_KEY_ML_COMPONENTS_PROMPT, ResearchKeyMLComponents, num_tool_calls=3)
print(ml_research_result, ml_research_tokens, ml_research_time)

In [ ]:
WRITE_MODEL_SPECIFICATION_PROMPT = f"""
Please design a model specification that will allow training and running a model on the {DEVICE}.
Take into account the hardware constraints researched by the Firmware Engineer and the project setup they have already done.
Also consider the key model features researched by the ML Scientist. Appropriate files based on the {DEVICE} toolkit.
This may include a PyTorch/other framework implementation with toolkit specific functions and/or sidecar files in config
languages such as YAML.

Firmware Engineer Notes:
Datasheet path: {hardware_research_result.datasheet_path}
Hardware constraints path: {hardware_research_result.hardware_constraints_path}
Inference code synthesis docs path: {hardware_research_result.inference_code_synthesis_docs_path}
Model specification docs path: {hardware_research_result.custom_model_specification_docs_path}
Custom dataset docs path: {hardware_research_result.custom_dataset_docs_path}
Examples path: {hardware_research_result.examples_path}
Summary: {hardware_research_result.summary}

ML Scientist notes:
Important features: {'; '.join(map(lambda f: f.description + ": " + f.code_snippet, ml_research_result.important_architecture_features))}
Notes: {ml_research_result.notes}

These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

model_specification_result, model_specification_tokens, model_specification_time = run_with_tools(ML_ENGINEER, WRITE_MODEL_SPECIFICATION_PROMPT, WriteModelSpecification, num_tool_calls=10)
print(model_specification_result, model_specification_tokens, model_specification_time)

In [ ]:
def VERIFY_ML_PROMPT(model_specification_result):
    return f"""
The ML Engineer has attempted to write a model specification. Please perform quick sanity checks to make sure it is not a buggy model.
Run a quick gradient check (compute loss+gradients for one example) to verify that they don't blow up/vanish.

ML Engineer results:
Compressed model architecture code path: {model_specification_result.compressed_model_architecture_code_path}
Side car file paths: {', '.join(model_specification_result.side_car_file_paths)}
Sample model input path: {model_specification_result.sample_model_input_path}
Randomly inititiated model weights path: {model_specification_result.randomly_initiated_model_weights_path}

These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

verify_ml_result, verify_ml_tokens, verify_ml_time = run_with_tools(ML_SCIENTIST, VERIFY_ML_PROMPT(model_specification_result), VerifyML, num_tool_calls=2)
print(verify_ml_result, verify_ml_tokens, verify_ml_time)

In [ ]:
def VERIFY_HARDWARE_PROMPT(model_specification_result):
    return f"""
The ML Engineer has attempted to write a model specification. Please perform quick sanity checks to make sure it is not a buggy model.
Run inference code synthesis checks to make sure that the ML model with random weights can run on the target hardware. 

ML Engineer results:
Compressed model architecture code path: {model_specification_result.compressed_model_architecture_code_path}
Side car file paths: {', '.join(model_specification_result.side_car_file_paths)}
Sample model input path: {model_specification_result.sample_model_input_path}
Randomly inititiated model weights path: {model_specification_result.randomly_initiated_model_weights_path}

These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

verify_hardware_result, verify_hardware_tokens, verify_hardware_time = run_with_tools(FIRMWARE_ENGINEER, VERIFY_HARDWARE_PROMPT(model_specification_result), VerifyHardwareConstraints, num_tool_calls=4)
print(verify_hardware_result, verify_hardware_tokens, verify_hardware_time)

In [ ]:
iterations = 1
verify_ml_tokens = 0
verify_ml_time = 0
verify_hardware_tokens = 0
verify_hardware_time = 0
while not verify_hardware_result.meets_all_constraints or not verify_ml_result.is_fully_valid_model:
    iterations += 1

    model_specification_result, tokens, time = run_with_tools(ML_ENGINEER, WRITE_MODEL_SPECIFICATION_PROMPT + f"""
        Current Firmware Engineer Feedback: {verify_hardware_result.feedback}   
        Current Firmware Engineer Error(s): {verify_hardware_result.errors}
        Current ML Scientist Feedback: {verify_ml_result.feedback}   
        Current ML Scientist Error(s): {verify_ml_result.errors}                                                   
    """, WriteModelSpecification, num_tool_calls=10)
    print(model_specification_result, tokens, time)
    model_specification_tokens += tokens
    model_specification_time += time

    verify_ml_result, tokens, time = run_with_tools(ML_SCIENTIST, VERIFY_ML_PROMPT(model_specification_result), VerifyML, num_tool_calls=2)
    print(verify_ml_result, tokens, time)
    verify_ml_tokens += tokens
    verify_ml_time += time

    verify_hardware_result, tokens, time = run_with_tools(FIRMWARE_ENGINEER, VERIFY_HARDWARE_PROMPT(model_specification_result), VerifyHardwareConstraints, num_tool_calls=4)
    print(verify_hardware_result, tokens, time)
    verify_hardware_tokens += tokens
    verify_hardware_time += time

In [ ]:
TRAIN_AND_EVALUATE_PROMPT = f"""
Train and evaluate the model specification developed by the ML Engineer using the dataset in {DATASET_PATH} for {TASK}.
Use the appropriate quantization and other tools to target the {DEVICE}.
If the model performs abnormally poorly, recommend improvements to the architecture.

ML Engineer results:
Compressed model architecture code path: {model_specification_result.compressed_model_architecture_code_path}
Side car file paths: {', '.join(model_specification_result.side_car_file_paths)}
Sample model input path: {model_specification_result.sample_model_input_path}
Randomly inititiated model weights path: {model_specification_result.randomly_initiated_model_weights_path}

These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

train_and_evaluate_result, train_and_evaluate_tokens, train_and_evaluate_time = run_with_tools(ML_SCIENTIST, TRAIN_AND_EVALUATE_PROMPT, TrainAndEvaluateModel, num_tool_calls=5)
print(train_and_evaluate_result, train_and_evaluate_tokens, train_and_evaluate_time)

In [ ]:
assert not train_and_evaluate_result.recommend_improvements_to_architecture, f"Please rerun model specification writing with notes: {train_and_evaluate_result.notes}"

In [ ]:
SYNTEHSIZE_PROMPT = f"""
Write/synthesize the inference code for {TASK} that can run on {DEVICE}.
Use the results from teh ML Engineer and Scientist. When you are done,
please summarize all the findings from the whole model compression process
and write that in the notes. Place instructions for flashing the built code
to the device here as well.

ML Engineer results:
Compressed model architecture code path: {model_specification_result.compressed_model_architecture_code_path}
Side car file paths: {', '.join(model_specification_result.side_car_file_paths)}
Sample model input path: {model_specification_result.sample_model_input_path}

ML Scientist Results
Trained weights path: {train_and_evaluate_result.trained_weights_path}
Notes: {train_and_evaluate_result.notes}

These are the files currently available in your working directory:
{list_files(playground.playground_path)}
""".strip()

synthesize_result, synthesize_tokens, synthesize_time = run_with_tools(FIRMWARE_ENGINEER, SYNTEHSIZE_PROMPT, FinalOutput, num_tool_calls=3)
print(synthesize_result, synthesize_tokens, synthesize_time)

In [ ]:
print("Process Done!!!")
total_time = hardware_research_time + ml_research_time + model_specification_time + verify_hardware_time + verify_ml_time + train_and_evaluate_time + synthesize_time # type: ignore
print("Total Time:", total_time, "seconds")
print("Evaluation Score:", train_and_evaluate_result.evaluation_score, train_and_evaluate_result.evaluation_metric)
print("Compressed Model Size:", synthesize_result.compressed_model_size_bytes / 1e3, "kb")
print("Compressed Model Path:", synthesize_result.compressed_model_path)
print("Inference Code Path:", synthesize_result.inference_code_path)
print("Notes:", synthesize_result.notes)